For the following data and scenarios (7-11):

   - a. Generate at least $R=2{,}000$ nonparametric bootstrap estimates with a reproducible RNG. Report the seed and $R$. For two groups, resample independently within groups; for correlation, resample paired rows.
   - b. Estimate the standard error $\widehat{SE}(\hat\theta)$ with the sample standard deviation of the bootstrap estimates (using denominator $R-1$, or `ddof=1`).
   - c. Construct centered-Normal 90%, 95%, and 99% CIs: $\hat\theta \pm z^*\widehat{SE}(\hat\theta)$.
   - d. Test the stated null using $z=(\hat\theta-\theta_0)/\widehat{SE}(\hat\theta)$ and $p=2\Phi(-|z|)$. Decide at $\alpha=.10,.05,.01$.
   - e. Confirm that every test decision agrees with whether the matching CI contains $\theta_0$.
   - f. Interpret the estimate, interval, and test in context. These are observational data: do not turn association into causation.

Use unrounded estimates and standard errors when computing intervals and tests. If a p-value is close to a decision threshold, increase $R$ (for example, to $100{,}000$, using batches if needed) to reduce Monte Carlo error. Small seed-dependent differences in results or borderline test decisions are acceptable when the procedure is correct and reproducible and you discuss the remaining simulation uncertainty. In particular, Problem 10 may be close to the 5% threshold.

# Question 7
7. For `./data/ames_prices.csv`, define an indicator equal to 1 when `Central.Air == 'Y'`. Estimate the proportion of homes with central air and test $H_0:p=.90$. Treat a proportion as the mean of a binary variable and complete the full inference routine.


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

In [3]:
df_ames_prices = pd.read_csv("ames_prices.csv")
df_ames_prices.head()


,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal


In [ ]:
#Question 7
#Making central air a binary variable then finding the mean of all the samples
df_ames_prices["air_conditioning"] = (df_ames_prices["Central.Air"] == "Y").astype(int)
n = len(df_ames_prices["air_conditioning"])
p_hat = df_ames_prices["air_conditioning"].mean()
print(f"Percentage of homes with central air: {p_hat}")

Percentage of homes with central air: 0.9331058020477816


In [ ]:
#Part A
#Bootstrap
np.random.seed(10)
R = 2_000
results = []
for t in range(R):
    idx = np.random.randint( 0 , len(df_ames_prices), len(df_ames_prices))
    p_hat_t = np.mean(df_ames_prices["air_conditioning"].iloc[idx])
    results.append(p_hat_t)
results = np.array(results)

print(results)

[0.92866894 0.92969283 0.9331058  ... 0.92764505 0.92969283 0.93447099]


In [ ]:
#Part B
#standard Error
se = results.std(ddof=1)
print("Estimated proportion:", p_hat)
print("Bootstrap standard error:", se)

Estimated proportion: 0.9331058020477816
Bootstrap standard error: 0.004671223725800887


In [ ]:
# Part c
# Nested confidence regions: largest first
# Getting the confidence intervals
confidence_intervals = []
for confidence, critical_value in [(0.99, 2.576),
                          (0.95, 1.960),
                          (0.90, 1.645)]:

    z = norm.ppf((1 + confidence) / 2)
    lower = p_hat - critical_value * se
    upper = p_hat + critical_value * se
    confidence_intervals.append((confidence, lower, upper)) #Do this to make it easy to store and use the bounds in future cell blocks
    print(f'{confidence} interval: [{lower}, {upper}]')
print(confidence_intervals)

0.99 interval: [0.9210727297301186, 0.9451388743654446]
0.95 interval: [0.9239502035452118, 0.9422614005503513]
0.9 interval: [0.9254216390188391, 0.9407899650767241]
[(0.99, np.float64(0.9210727297301186), np.float64(0.9451388743654446)), (0.95, np.float64(0.9239502035452118), np.float64(0.9422614005503513)), (0.9, np.float64(0.9254216390188391), np.float64(0.9407899650767241))]


In [ ]:
# Part d 
# Collecting the confidence Intervals 
# Comparing values to test the Hypothesis 
null_hypothesis = 0.90

z_stat = (p_hat - null_hypothesis) / se
p_value = 2 * norm.cdf(-abs(z_stat))

print("z-statistic:", z_stat)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    decision = (
        "Reject H0" if p_value < alpha
        else "Fail to reject H0"
    )
    print(f"alpha = {alpha}: {decision}")

z-statistic: 7.087179718009662
p-value: 1.368725913052248e-12
alpha = 0.1: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


In [ ]:
# Part E
# Automatically checks if the the null H0 z-score is in any of the confidence intervals
for level, lower, upper in confidence_intervals:
    p_hat - critical_value * se
    contains_null = lower <= null_hypothesis <= upper
    alpha = 1 - level

    reject_h0 = p_value < alpha

    print(
        f"{level} CI contains 0.90: {contains_null}; "
        f"Reject H0 at alpha={alpha:.2f}: {reject_h0}", #Need to round alpha because it produces long decimals
    )

0.99 CI contains 0.90: False; Reject H0 at alpha=0.01: True
0.95 CI contains 0.90: False; Reject H0 at alpha=0.05: True
0.9 CI contains 0.90: False; Reject H0 at alpha=0.10: True


Answer: About 93% of homes has central air. The Standard error between sample and H0 was 0.459%. The confidence intervals were 
0.99 interval: [0.9212782015751199, 0.9449334025204433]
0.95 interval: [0.9241065408185825, 0.9421050632769807]
0.9 interval: [0.9255528506589895, 0.9406587534365737].
At all levels of alpha, we reject the null H0 and none of the confidence intervals contain H0. 

# Question 8
8. For `./data/ames_prices.csv`, estimate the Pearson correlation between `arcsinh(area)` and `arcsinh(price)`. Bootstrap paired rows, construct all three CIs, and test $H_0:\rho=0$. Explain what would go wrong if area and price were resampled separately.

In [31]:
x = np.arcsinh(df_ames_prices["area"])
y = np.arcsinh(df_ames_prices["price"])

# Original Pearson correlation estimate
rho_hat = np.corrcoef(x, y)[0, 1]

print("Estimated correlation:", rho_hat)

Estimated correlation: 0.7231661530091357


In [32]:
#Part A
#Bootstrap
np.random.seed(10)
R = 2_000
results = []
for t in range(R):
    idx = np.random.randint( 0 , len(df_ames_prices), len(df_ames_prices))
    r_hat_t = np.corrcoef(x.iloc[idx], y.iloc[idx])[0,1]
    results.append(r_hat_t)
results = np.array(results)

print(results)

[0.72448185 0.71256965 0.73639984 ... 0.7291966  0.70764769 0.71013573]


In [ ]:
#Part B
#standard Error
se = results.std(ddof=1)
print("Estimated proportion:", r_hat_t)
print("Bootstrap standard error:", se)

Estimated proportion: 0.9331058020477816
Bootstrap standard error: 0.010010037598944163


In [33]:
# Part c
# Nested confidence regions: largest first
# Getting the confidence intervals
confidence_intervals = []
for confidence, critical_value in [(0.99, 2.576),
                          (0.95, 1.960),
                          (0.90, 1.645)]:

    z = norm.ppf((1 + confidence) / 2)
    lower = r_hat_t - critical_value * se
    upper = r_hat_t + critical_value * se
    confidence_intervals.append((confidence, lower, upper)) #Do this to make it easy to store and use the bounds in future cell blocks
    print(f'{confidence} interval: [{lower}, {upper}]')
print(confidence_intervals)

0.99 interval: [0.6843498756465141, 0.7359215893562745]
0.95 interval: [0.6905160588074637, 0.7297554061953249]
0.9 interval: [0.6936692206511311, 0.7266022443516574]
[(0.99, np.float64(0.6843498756465141), np.float64(0.7359215893562745)), (0.95, np.float64(0.6905160588074637), np.float64(0.7297554061953249)), (0.9, np.float64(0.6936692206511311), np.float64(0.7266022443516574))]


In [37]:
# Part d 
# Collecting the confidence Intervals 
# Comparing values to test the Hypothesis 
null_hypothesis = 0

z_stat = (r_hat_t - null_hypothesis) / se
p_value = 2 * norm.cdf(-abs(z_stat))

print("z-statistic:", z_stat)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    decision = (
        "Reject H0" if p_value < alpha
        else "Fail to reject H0"
    )
    print(f"alpha = {alpha}: {decision}")

z-statistic: 70.94236415019039
p-value: 0.0
alpha = 0.1: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


In [38]:
# Part E
# Automatically checks if the the null H0 z-score is in any of the confidence intervals
for level, lower, upper in confidence_intervals:
    p_hat - critical_value * se
    contains_null = lower <= null_hypothesis <= upper
    alpha = 1 - level

    reject_h0 = p_value < alpha

    print(
        f"{level} CI contains 0.90: {contains_null}; "
        f"Reject H0 at alpha={alpha:.2f}: {reject_h0}", #Need to round alpha because it produces long decimals
    )

0.99 CI contains 0.90: False; Reject H0 at alpha=0.01: True
0.95 CI contains 0.90: False; Reject H0 at alpha=0.05: True
0.9 CI contains 0.90: False; Reject H0 at alpha=0.10: True


Their is a positive association between home area and home price. Area and price must be resampled as paired rows because each home's area is associated with its own price. Resampling them separately would break these pairings and produce an incorrect bootstrap distribution.

# Question 9
9. For `./data/metabric.csv`, use `Overall Survival (Months)` to estimate the mean recorded time to death or last follow-up, including both living and deceased patients. Let $\mu$ denote the population mean of this recorded-time variable, and test $H_0:\mu=120$ months. Complete the full inference routine and interpret the CIs in both months and years. Living patients' eventual death times are right-censored, so this exercise does not estimate unrestricted mean lifetime. Explain that distinction in your interpretation.

In [26]:
df_metabric = pd.read_csv("./metabric.csv")

# Recorded time to death or last follow-up, in months
x = df_metabric["Overall Survival (Months)"].dropna().to_numpy()

mean_hat = x.mean()

print("Estimated mean recorded time (months):", mean_hat)
df_metabric.head()

Estimated mean recorded time (months): 129.09429138744105


,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Chemotherapy,Hormone Therapy,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Overall Survival (Months),Overall Survival Status,Radio Therapy,TMB (nonsynonymous),Tumor Size,Tumor Stage
0,43.19,BREAST CONSERVING,Breast Cancer,NO,YES,0.0,2.0,4.020,84.633333,0:LIVING,YES,2.615035,10.0,1.0
1,48.87,MASTECTOMY,Breast Cancer,YES,YES,1.0,2.0,4.030,163.700000,1:DECEASED,NO,2.615035,15.0,2.0
2,47.68,MASTECTOMY,Breast Cancer,YES,YES,3.0,1.0,4.050,164.933333,0:LIVING,YES,1.307518,25.0,2.0
3,76.97,MASTECTOMY,Breast Cancer,YES,YES,8.0,2.0,6.080,41.366667,1:DECEASED,YES,2.615035,40.0,2.0
4,78.77,MASTECTOMY,Breast Cancer,NO,YES,0.0,4.0,4.062,7.800000,1:DECEASED,YES,5.230071,31.0,4.0


In [40]:
#Part A
#Bootstrap
np.random.seed(10)
R = 2_000
results = []
for t in range(R):
    idx = np.random.randint( 0 , len(df_metabric), len(df_metabric))
    mean_hat_t = np.mean(df_metabric['Overall Survival (Months)'].iloc[idx])
    results.append(mean_hat_t)
results = np.array(results)

print(results)

[127.847034   132.15063291 127.68796227 ... 128.08592703 125.77532887
 126.80541077]


In [41]:
#Part B
#standard Error
se = results.std(ddof=1)
print("Estimated proportion:", mean_hat_t)
print("Bootstrap standard error:", se)

Estimated proportion: 126.80541077190371
Bootstrap standard error: 2.131840165123921


In [42]:
# Part c
# Nested confidence regions: largest first
# Getting the confidence intervals
confidence_intervals = []
for confidence, critical_value in [(0.99, 2.576),
                          (0.95, 1.960),
                          (0.90, 1.645)]:

    z = norm.ppf((1 + confidence) / 2)
    lower = mean_hat_t - critical_value * se
    upper = mean_hat_t + critical_value * se
    confidence_intervals.append((confidence, lower, upper)) #Do this to make it easy to store and use the bounds in future cell blocks
    print(f'{confidence} interval: [{lower}, {upper}]')
print(confidence_intervals)

0.99 interval: [121.31379050654449, 132.29703103726294]
0.95 interval: [122.62700404826083, 130.9838174955466]
0.9 interval: [123.29853370027486, 130.31228784353257]
[(0.99, np.float64(121.31379050654449), np.float64(132.29703103726294)), (0.95, np.float64(122.62700404826083), np.float64(130.9838174955466)), (0.9, np.float64(123.29853370027486), np.float64(130.31228784353257))]


In [43]:
# Part d 
# Collecting the confidence Intervals 
# Comparing values to test the Hypothesis 
null_hypothesis = 120.0

z_stat = (mean_hat_t - null_hypothesis) / se
p_value = 2 * norm.cdf(-abs(z_stat))

print("z-statistic:", z_stat)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    decision = (
        "Reject H0" if p_value < alpha
        else "Fail to reject H0"
    )
    print(f"alpha = {alpha}: {decision}")

z-statistic: 3.1922706416914326
p-value: 0.0014115900111241998
alpha = 0.1: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


In [ ]:
# Part E
# Automatically checks if the the null H0 z-score is in any of the confidence intervals
for level, lower, upper in confidence_intervals:
    mean_hat_t - critical_value * se
    contains_null = lower <= null_hypothesis <= upper
    alpha = 1 - level

    reject_h0 = p_value < alpha

    print(
        f"{level} CI contains 0.90: {contains_null}; "
        f"Reject H0 at alpha={alpha:.2f}: {reject_h0}", #Need to round alpha because it produces long decimals
    )

0.99 CI contains 0.90: False; Reject H0 at alpha=0.01: True
0.95 CI contains 0.90: False; Reject H0 at alpha=0.05: True
0.9 CI contains 0.90: False; Reject H0 at alpha=0.10: True


The results shows evidence that the population mean recorded time to death or last follow-up is more than 120 months. For patients that are still alive, the recorded time ends at their last follow-up, so their eventual death times are right-censored and not shown. Therefore, the estimated mean only describes recorded observation times, not the average time until death for the entire population.

# Question 10
10. For `./data/metabric.csv`, create `deceased = 1` when `Overall Survival Status == '1:DECEASED'`. Estimate the difference in deceased proportions between `Chemotherapy == 'YES'` and `'NO'`: YES minus NO. Test $H_0:\Delta=0$. State at least two plausible confounders visible in the dataset.

One plausible confounder is age at because if are typically going to have a shorter lifespan and more difficult time healing if you are old rather than young. Another plausible confounder is the tumor stage because your are less likely to survive if you have a high stage tumor rather then an tumor that had just formed

In [63]:

df_metabric["deceased"] = (df_metabric["Overall Survival Status"] == "1:DECEASED").astype(int)

# Separate patients by chemotherapy status
yes = df_metabric.loc[df_metabric["Chemotherapy"] == "YES", "deceased"]

no = df_metabric.loc[df_metabric["Chemotherapy"] == "NO", "deceased"]

# Calculate deceased proportions
p_yes = yes.mean()
p_no = no.mean()
# differnce between the two statistics 
delta_hat = p_yes - p_no

In [64]:
#Part A
#Bootstrap
np.random.seed(10)
R = 2_000
results = []
for t in range(R):
    idx_yes = np.random.randint(0, len(yes), len(yes))
    idx_no = np.random.randint(0, len(no), len(no))
    p_yes_t = np.mean(yes.iloc[idx_yes])
    p_no_t = np.mean(no.iloc[idx_no])
    temp = p_yes_t - p_no_t
    results.append(temp)
results = np.array(results)

print(results)

[-0.06340679 -0.04366164 -0.05082004 ... -0.06130955 -0.02741298
 -0.05661888]


In [65]:
#Part B
#standard Error
se = results.std(ddof=1)
print("Estimated proportion:", delta_hat)
print("Bootstrap standard error:", se)

Estimated proportion: -0.0646406573558892
Bootstrap standard error: 0.033265040810365805


In [66]:
# Part c
# Nested confidence regions: largest first
# Getting the confidence intervals
confidence_intervals = []
for confidence, critical_value in [(0.99, 2.576),
                          (0.95, 1.960),
                          (0.90, 1.645)]:

    z = norm.ppf((1 + confidence) / 2)
    lower = delta_hat - critical_value * se
    upper = delta_hat + critical_value * se
    confidence_intervals.append((confidence, lower, upper)) #Do this to make it easy to store and use the bounds in future cell blocks
    print(f'{confidence} interval: [{lower}, {upper}]')
print(confidence_intervals)

0.99 interval: [-0.1503314024833915, 0.021050087771613116]
0.95 interval: [-0.12984013734420619, 0.0005588226324277773]
0.9 interval: [-0.11936164948894094, -0.009919665222837451]
[(0.99, np.float64(-0.1503314024833915), np.float64(0.021050087771613116)), (0.95, np.float64(-0.12984013734420619), np.float64(0.0005588226324277773)), (0.9, np.float64(-0.11936164948894094), np.float64(-0.009919665222837451))]


In [67]:
# Part d 
# Collecting the confidence Intervals 
# Comparing values to test the Hypothesis 
null_hypothesis = 0

z_stat = (delta_hat - null_hypothesis) / se
p_value = 2 * norm.cdf(-abs(z_stat))

print("z-statistic:", z_stat)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    decision = (
        "Reject H0" if p_value < alpha
        else "Fail to reject H0"
    )
    print(f"alpha = {alpha}: {decision}")

z-statistic: -1.9432009034465503
p-value: 0.0519918889866679
alpha = 0.1: Reject H0
alpha = 0.05: Fail to reject H0
alpha = 0.01: Fail to reject H0


In [68]:
# Part E
# Automatically checks if the the null H0 z-score is in any of the confidence intervals
for level, lower, upper in confidence_intervals:
    mean_hat_t - critical_value * se
    contains_null = lower <= null_hypothesis <= upper
    alpha = 1 - level

    reject_h0 = p_value < alpha

    print(
        f"{level} CI contains 0.90: {contains_null}; "
        f"Reject H0 at alpha={alpha:.2f}: {reject_h0}", #Need to round alpha because it produces long decimals
    )

0.99 CI contains 0.90: True; Reject H0 at alpha=0.01: False
0.95 CI contains 0.90: True; Reject H0 at alpha=0.05: False
0.9 CI contains 0.90: False; Reject H0 at alpha=0.10: True


# Question 11
11. For `./data/nhanes_data_17_18.csv`, use `AgeInYearsAtScreening` to select respondents aged 18 or older with observed `SleepHoursWeekdaysOrWorkdays`. Estimate mean weekday sleep and test $H_0:\mu=8$ hours. Comment on whether statistical and practical significance tell the same story. For this exercise, treat observations as independent and unweighted, ignoring the complex survey design. Do not interpret these results as nationally representative population estimates or survey-adjusted confidence intervals.

The results of these calculation is that the observed people sleep less than 8 hours a day. Statistical significance and practical significance do not rely the same message in this problem. This data is not repsentative of everyone because some people will naturally need more sleep than others.

In [51]:

df_nhanes = pd.read_csv('./nhanes_data_17_18.csv')
df_nhanes_filter = df_nhanes.loc[df_nhanes['AgeInYearsAtScreening'] >= 18, ['SleepHoursWeekdaysOrWorkdays']].dropna()
sleep_hat = np.mean(df_nhanes_filter['SleepHoursWeekdaysOrWorkdays'])
print(sleep_hat)

7.62124248496994


C:\Users\jake_\AppData\Local\Temp\ipykernel_19560\259201019.py:1: DtypeWarning: Columns (142: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nhanes = pd.read_csv('./nhanes_data_17_18.csv')


In [54]:
#Part A
#Bootstrap
np.random.seed(10)
R = 2_000
results = []
for t in range(R):
    idx = np.random.randint( 0 , len(df_nhanes_filter), len(df_nhanes_filter))
    temp = np.mean(df_nhanes['SleepHoursWeekdaysOrWorkdays'].iloc[idx])
    results.append(temp)
results = np.array(results)

print(results)

[7.60662148 7.66114733 7.66758458 ... 7.62342105 7.62070773 7.63899059]


In [55]:
#Part B
#standard Error
se = results.std(ddof=1)
print("Estimated proportion:", sleep_hat)
print("Bootstrap standard error:", se)

Estimated proportion: 7.62124248496994
Bootstrap standard error: 0.02702365196047316


In [56]:
# Part c
# Nested confidence regions: largest first
# Getting the confidence intervals
confidence_intervals = []
for confidence, critical_value in [(0.99, 2.576),
                          (0.95, 1.960),
                          (0.90, 1.645)]:

    z = norm.ppf((1 + confidence) / 2)
    lower = sleep_hat - critical_value * se
    upper = sleep_hat + critical_value * se
    confidence_intervals.append((confidence, lower, upper)) #Do this to make it easy to store and use the bounds in future cell blocks
    print(f'{confidence} interval: [{lower}, {upper}]')
print(confidence_intervals)

0.99 interval: [7.551629557519761, 7.690855412420119]
0.95 interval: [7.568276127127413, 7.674208842812468]
0.9 interval: [7.576788577494962, 7.665696392444919]
[(0.99, np.float64(7.551629557519761), np.float64(7.690855412420119)), (0.95, np.float64(7.568276127127413), np.float64(7.674208842812468)), (0.9, np.float64(7.576788577494962), np.float64(7.665696392444919))]


In [58]:
# Part d 
# Collecting the confidence Intervals 
# Comparing values to test the Hypothesis 
null_hypothesis = 8

z_stat = (sleep_hat - null_hypothesis) / se
p_value = 2 * norm.cdf(-abs(z_stat))

print("z-statistic:", z_stat)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    decision = (
        "Reject H0" if p_value < alpha
        else "Fail to reject H0"
    )
    print(f"alpha = {alpha}: {decision}")

z-statistic: -14.015778310942547
p-value: 1.248222881938544e-44
alpha = 0.1: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


In [59]:
# Part E
# Automatically checks if the the null H0 z-score is in any of the confidence intervals
for level, lower, upper in confidence_intervals:
    sleep_hat - critical_value * se
    contains_null = lower <= null_hypothesis <= upper
    alpha = 1 - level

    reject_h0 = p_value < alpha

    print(
        f"{level} CI contains 0.90: {contains_null}; "
        f"Reject H0 at alpha={alpha:.2f}: {reject_h0}", #Need to round alpha because it produces long decimals
    )

0.99 CI contains 0.90: False; Reject H0 at alpha=0.01: True
0.95 CI contains 0.90: False; Reject H0 at alpha=0.05: True
0.9 CI contains 0.90: False; Reject H0 at alpha=0.10: True
